Tokenization of MIDI Data files

Token Vocabulary:

1. TIME_STEP_SECONDS (rounded to 10ms steps)
2. MAX_TIME_SHIFT (advances time by at most 100 steps = 1 second)
3. VELOCITY_BINS (compressed into 16 categories)
4. NOTE_COUNT (all 128 note pitches)

| Token type | Count | Meaning |
|---|---:|---|
| `NOTE_ON_0` … `NOTE_ON_127` | 128 | Note is ON |
| `NOTE_OFF_0` … `NOTE_OFF_127` | 128 | Note is OFF |
| `TIME_SHIFT_1` … `TIME_SHIFT_100` | 100 | Advance time by 0.01–1.00 seconds |
| `VELOCITY_0` … `VELOCITY_15` | 16 | Set the velocity category |
| `PEDAL_ON`, `PEDAL_OFF` | 2 | Change sustain pedal state |
| `BOS`, `EOS` | 2 | Mark the beginning and end of a piece |
| **Total** | **376** | |

In [ ]:
from pathlib import Path
import csv, json, pickle
import numpy as np
import pandas as pd
import pretty_midi

DATA_ROOT = Path("maestro-data")
OUT_DIR = Path("data/maestro")
CSV_PATH = DATA_ROOT/"maestro-v3.0.0.csv"
assert CSV_PATH.is_file(), f"Metadata CSV not found: {CSV_PATH.resolve()}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

Dataset: /Users/chloeheh/Documents/DSA4212/DSA4212-project/maestro-data
Output: /Users/chloeheh/Documents/DSA4212/DSA4212-project/data/maestro


In [ ]:
metadata = pd.read_csv(CSV_PATH)
display(metadata[["year", "split", "midi_filename"]].head())
print(metadata["split"].value_counts())

assert set(metadata["split"]) == {"train", "validation", "test"}

missing = [str(DATA_ROOT / p) for p in metadata["midi_filename"] if not (DATA_ROOT / p).is_file()]

assert not missing, f"Missing {len(missing)} MIDI files, e.g. {missing[:3]}"
print("All MIDI file paths exist.")

,year,split,midi_filename
0,2018,train,2018/MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R...
1,2008,train,2008/MIDI-Unprocessed_03_R2_2008_01-03_ORIG_MI...
2,2017,train,2017/MIDI-Unprocessed_066_PIANO066_MID--AUDIO-...
3,2004,train,2004/MIDI-Unprocessed_XP_21_R1_2004_01_ORIG_MI...
4,2006,validation,2006/MIDI-Unprocessed_17_R1_2006_01-06_ORIG_MI...


split
train         962
test          177
validation    137
Name: count, dtype: int64
All MIDI file paths exist.


In [1]:
TIME_STEP_SECONDS = 0.01
MAX_TIME_SHIFT = 100
VELOCITY_BINS = 16
NOTE_COUNT = 128

TOKENS = ([f"NOTE_ON_{p}" for p in range(NOTE_COUNT)]
          + [f"NOTE_OFF_{p}" for p in range(NOTE_COUNT)]
          + [f"TIME_SHIFT_{i}" for i in range(1, MAX_TIME_SHIFT + 1)] # Start from 1 ms
          + [f"VELOCITY_{i}" for i in range(VELOCITY_BINS)]
          + ["PEDAL_ON", "PEDAL_OFF", "BOS", "EOS"])
STOI = {token: i for i, token in enumerate(TOKENS)}
ITOS = {i: token for token, i in STOI.items()}
assert len(TOKENS) == 376

In [5]:
def velocity_to_bin(v):
    return min(VELOCITY_BINS - 1, max(0, (int(v) * VELOCITY_BINS) // 128))

def bin_to_velocity(b):
    return min(127, max(1, round((b + 0.5) * 128 / VELOCITY_BINS)))

def midi_to_tokens(path):
    """Encode notes and CC64 sustain pedal, treating MIDI time in seconds.

    Other MIDI data (tempo/program/CC66/CC67) is intentionally omitted.
    Note-offs are emitted before note-ons at the same quantized tick.
    """
    midi = pretty_midi.PrettyMIDI(str(path))
    events = []
    for inst in midi.instruments:
        if inst.is_drum:
            continue
        for note in inst.notes:
            start = max(0, round(note.start / TIME_STEP_SECONDS))
            end = max(start + 1, round(note.end / TIME_STEP_SECONDS))
            # priority: note-off=0; pedal=1; note-on=2
            events.append((end, 0, note.pitch, 0))
            events.append((start, 2, note.pitch, note.velocity))
        for cc in inst.control_changes:
            if cc.number == 64:
                tick = max(0, round(cc.time / TIME_STEP_SECONDS))
                events.append((tick, 1, 1 if cc.value >= 64 else 0, 0))
    events.sort(key=lambda x: (x[0], x[1], x[2]))
    ids = [STOI["BOS"]]
    prev_tick = 0
    last_pedal = False
    for tick, kind, pitch_or_flag, velocity in events:
        delta = tick - prev_tick
        while delta > 0:
            step = min(delta, MAX_TIME_SHIFT)
            ids.append(STOI[f"TIME_SHIFT_{step}"])
            delta -= step
        prev_tick = tick
        if kind == 0:
            ids.append(STOI[f"NOTE_OFF_{pitch_or_flag}"])
        elif kind == 1:
            pedal = bool(pitch_or_flag)
            if pedal != last_pedal:
                ids.append(STOI["PEDAL_ON" if pedal else "PEDAL_OFF"])
                last_pedal = pedal
        else:
            ids.append(STOI[f"VELOCITY_{velocity_to_bin(velocity)}"])
            ids.append(STOI[f"NOTE_ON_{pitch_or_flag}"])
    ids.append(STOI["EOS"])
    return ids

def tokens_to_midi(ids):
    """Decode to an approximate performance, for basic inspection and testing.

    Multiple same-pitch overlapping notes are handled with FIFO matching. This
    tokenizer is intentionally lossy due to timing/velocity quantization.
    """
    midi = pretty_midi.PrettyMIDI()
    inst = pretty_midi.Instrument(program=0, name="Piano")
    active = {p: [] for p in range(NOTE_COUNT)}
    tick, velocity, pedal = 0, 64, False
    for token_id in ids:
        t = ITOS[int(token_id)]
        if t == "BOS":
            continue
        if t == "EOS":
            break
        if t.startswith("TIME_SHIFT_"):
            tick += int(t.split("_")[-1])
        elif t.startswith("VELOCITY_"):
            velocity = bin_to_velocity(int(t.split("_")[-1]))
        elif t.startswith("NOTE_ON_"):
            p = int(t.split("_")[-1])
            active[p].append((tick, velocity))
        elif t.startswith("NOTE_OFF_"):
            p = int(t.split("_")[-1])
            if active[p]:
                start, vel = active[p].pop(0)
                inst.notes.append(pretty_midi.Note(
                    velocity=vel, pitch=p,
                    start=start * TIME_STEP_SECONDS,
                    end=max(tick, start + 1) * TIME_STEP_SECONDS))
        elif t in ("PEDAL_ON", "PEDAL_OFF"):
            is_on = (t == "PEDAL_ON")
            if pedal != is_on:
                inst.control_changes.append(pretty_midi.ControlChange(
                    number=64, value=127 if is_on else 0,
                    time=tick * TIME_STEP_SECONDS))
                pedal = is_on
    # Terminate any dangling notes for generated sequences.
    for p, starts in active.items():
        for start, vel in starts:
            inst.notes.append(pretty_midi.Note(
                velocity=vel, pitch=p,
                start=start * TIME_STEP_SECONDS,
                end=max(tick, start + 1) * TIME_STEP_SECONDS))
    midi.instruments.append(inst)
    return midi

In [6]:
example_row = metadata[metadata["split"] == "train"].iloc[0]
example_path = DATA_ROOT / example_row["midi_filename"]
example_ids = midi_to_tokens(example_path)
print("Example:", example_path.name)
print("Token count:", len(example_ids))
print("First 50 tokens:")
print([ITOS[i] for i in example_ids[:50]])
assert example_ids[0] == STOI["BOS"] and example_ids[-1] == STOI["EOS"]
roundtrip = OUT_DIR / "roundtrip_example.mid"
tokens_to_midi(example_ids).write(str(roundtrip))
print("Reconstructed MIDI saved:", roundtrip.resolve())

Example: MIDI-Unprocessed_Chamber3_MID--AUDIO_10_R3_2018_wav--1.midi
Token count: 32684
First 50 tokens:
['BOS', 'PEDAL_ON', 'TIME_SHIFT_98', 'VELOCITY_6', 'NOTE_ON_67', 'TIME_SHIFT_80', 'VELOCITY_8', 'NOTE_ON_72', 'TIME_SHIFT_3', 'NOTE_OFF_67', 'TIME_SHIFT_10', 'NOTE_OFF_72', 'TIME_SHIFT_16', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'VELOCITY_5', 'NOTE_ON_71', 'VELOCITY_8', 'NOTE_ON_78', 'TIME_SHIFT_2', 'TIME_SHIFT_1', 'VELOCITY_4', 'NOTE_ON_61', 'VELOCITY_4', 'NOTE_ON_67', 'TIME_SHIFT_1', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_3', 'TIME_SHIFT_2', 'PEDAL_OFF', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_12', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'TIME_SHIFT_2', 'PEDAL_ON', 'TIME_SHIFT_2', 'TIME_SHIFT_1', 'TIME_SHIFT_2', 'TIME_SHIFT_3']
Reconstructed MIDI saved: /Users/chloeheh/Documents/DSA4212/DSA4212-project/data/maestro/roundtrip_example.mid


In [7]:
manifest = {}
counts = {}
for split, output_name in [("train", "train"), ("validation", "val"), ("test", "test")]:
    rows = metadata[metadata["split"] == split]
    offsets, total_tokens = [], 0
    with (OUT_DIR / f"{output_name}.bin").open("wb") as f:
        for ix, (_, row) in enumerate(rows.iterrows(), start=1):
            ids = midi_to_tokens(DATA_ROOT / row["midi_filename"])
            np.asarray(ids, dtype=np.uint16).tofile(f)
            offsets.append({"midi_filename": row["midi_filename"],
                            "start": total_tokens, "length": len(ids),
                            "duration_seconds": float(row["duration"])})
            total_tokens += len(ids)
            if ix % 100 == 0:
                print(f"{split}: processed {ix}/{len(rows)}")
    manifest[split] = offsets
    counts[split] = {"pieces": int(len(rows)), "tokens": int(total_tokens),
                     "mean_tokens_per_piece": float(total_tokens / len(rows))}
    print(f"Finished {split}: {len(rows)} pieces, {total_tokens:,} tokens")

meta = {"vocab_size": len(TOKENS), "stoi": STOI, "itos": ITOS,
        "time_step_seconds": TIME_STEP_SECONDS, "max_time_shift": MAX_TIME_SHIFT,
        "velocity_bins": VELOCITY_BINS, "counts": counts,
        "tokenizer": "MAESTRO-v3-event-v1"}
with (OUT_DIR / "meta.pkl").open("wb") as f:
    pickle.dump(meta, f)
(OUT_DIR / "piece_offsets.json").write_text(json.dumps(manifest, indent=2))
(OUT_DIR / "summary.json").write_text(json.dumps(counts, indent=2))
print("All outputs saved in:", OUT_DIR.resolve())

train: processed 100/962
train: processed 200/962
train: processed 300/962
train: processed 400/962
train: processed 500/962
train: processed 600/962
train: processed 700/962
train: processed 800/962
train: processed 900/962
Finished train: 962 pieces, 33,293,600 tokens
validation: processed 100/137
Finished validation: 137 pieces, 3,790,187 tokens
test: processed 100/177
Finished test: 177 pieces, 4,334,269 tokens
All outputs saved in: /Users/chloeheh/Documents/DSA4212/DSA4212-project/data/maestro


In [8]:
for split, name in [("train", "train"), ("validation", "val"), ("test", "test")]:
    arr = np.memmap(OUT_DIR / f"{name}.bin", dtype=np.uint16, mode="r")
    assert len(arr) == counts[split]["tokens"]
    assert int(arr.max()) < len(TOKENS)
    assert len(manifest[split]) == counts[split]["pieces"]
    assert all(int(arr[p["start"]]) == STOI["BOS"] and
               int(arr[p["start"] + p["length"] - 1]) == STOI["EOS"]
               for p in manifest[split])
    print(f"{split:>10}: {len(arr):>12,} tokens; {len(manifest[split]):>4} pieces")
display(pd.DataFrame(counts).T)
print("Tokenizer complete. Next step: a piece-aware nanoGPT DataLoader.")

     train:   33,293,600 tokens;  962 pieces
validation:    3,790,187 tokens;  137 pieces
      test:    4,334,269 tokens;  177 pieces


,pieces,tokens,mean_tokens_per_piece
train,962.0,33293600.0,34608.731809
validation,137.0,3790187.0,27665.598540
test,177.0,4334269.0,24487.395480


Tokenizer complete. Next step: a piece-aware nanoGPT DataLoader.
